In [1]:
import pandas as pd

In [2]:
folderPath = "UNSW-NB15/"

In [3]:
dataDF = pd.read_csv(f"{folderPath}UNSW-NB15_1WithLables.csv")
dataDF.head()

/tmp/ipykernel_54356/3683186457.py:1: DtypeWarning: Columns (2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  dataDF = pd.read_csv(f"{folderPath}UNSW-NB15_1WithLables.csv")


,Unnamed: 0,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,...,0,3,7,1,3,1,1,1,Normal,0
1,1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,...,0,2,4,2,3,1,1,2,Normal,0
2,2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,...,0,12,8,1,2,2,1,1,Normal,0
3,3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,...,0,6,9,1,1,1,1,1,Normal,0
4,4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,...,0,7,9,1,1,1,1,1,Normal,0


In [4]:
dataDF.columns

Index(['Unnamed: 0', 'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state',
       'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service',
       'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb',
       'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit',
       'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat',
       'Label'],
      dtype='object')

In [5]:
required_cols = [
    "srcip",
    "dstip",
    "sport",
    "dsport",
    "proto",
    "state",
    "dur",
    "sbytes",
    "dbytes",
    "Spkts",
    "Dpkts",
    "Sload",
    "Dload",
    "sttl",
    "dttl",
    "Label"
]

dataDF = dataDF[required_cols].copy()

In [6]:
dataDF.head()

,srcip,dstip,sport,dsport,proto,state,dur,sbytes,dbytes,Spkts,Dpkts,Sload,Dload,sttl,dttl,Label
0,59.166.0.0,149.171.126.6,1390,53,udp,CON,0.001055,132,164,2,2,500473.93750,621800.93750,31,29,0
1,59.166.0.0,149.171.126.9,33661,1024,udp,CON,0.036133,528,304,4,4,87676.08594,50480.17188,31,29,0
2,59.166.0.6,149.171.126.7,1464,53,udp,CON,0.001119,146,178,2,2,521894.53130,636282.37500,31,29,0
3,59.166.0.5,149.171.126.5,3593,53,udp,CON,0.001209,132,164,2,2,436724.56250,542597.18750,31,29,0
4,59.166.0.3,149.171.126.0,49664,53,udp,CON,0.001169,146,178,2,2,499572.25000,609067.56250,31,29,0


In [7]:
# Convert to lowercase and remove leading/trailing spaces
dataDF['proto'] = (
    dataDF['proto']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Remove internal spaces (e.g., "u dp" -> "udp")
dataDF['proto'] = dataDF['proto'].str.replace(r'\s+', '', regex=True)

In [8]:
# Check for missing values
print(dataDF['proto'].isna().sum())

# Check for empty strings
print((dataDF['proto'] == '').sum())

# Check for leading/trailing whitespace (should be 0)
print((dataDF['proto'] != dataDF['proto'].str.strip()).sum())

0
0
0


In [9]:
dataDF['proto'].value_counts()

proto
tcp         160615
udp          74274
unas          1442
arp           1394
ospf           832
             ...  
sccopmce        14
ib              14
udt              2
esp              2
rtp              1
Name: count, Length: 135, dtype: int64

In [10]:
# Remove unwanted spaces in strings
dataDF["proto"] = dataDF["proto"].str.strip()
dataDF["state"] = dataDF["state"].str.strip()

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

proto_encoder = LabelEncoder()
state_encoder = LabelEncoder()

dataDF["proto"] = proto_encoder.fit_transform(dataDF["proto"])
dataDF["state"] = state_encoder.fit_transform(dataDF["state"])

# Convert numeric columns
numeric_cols = [
    "sport",
    "dsport",
    "dur",
    "sbytes",
    "dbytes",
    "Spkts",
    "Dpkts",
    "Sload",
    "Dload",
    "sttl",
    "dttl"
]

for col in numeric_cols:
    dataDF[col] = pd.to_numeric(dataDF[col], errors="coerce")

dataDF[numeric_cols] = dataDF[numeric_cols].fillna(0)

In [11]:
dataDF.head()

,srcip,dstip,sport,dsport,proto,state,dur,sbytes,dbytes,Spkts,Dpkts,Sload,Dload,sttl,dttl,Label
0,59.166.0.0,149.171.126.6,1390.0,53.0,120,2,0.001055,132,164,2,2,500473.93750,621800.93750,31,29,0
1,59.166.0.0,149.171.126.9,33661.0,1024.0,120,2,0.036133,528,304,4,4,87676.08594,50480.17188,31,29,0
2,59.166.0.6,149.171.126.7,1464.0,53.0,120,2,0.001119,146,178,2,2,521894.53130,636282.37500,31,29,0
3,59.166.0.5,149.171.126.5,3593.0,53.0,120,2,0.001209,132,164,2,2,436724.56250,542597.18750,31,29,0
4,59.166.0.3,149.171.126.0,49664.0,53.0,120,2,0.001169,146,178,2,2,499572.25000,609067.56250,31,29,0


In [12]:
dataDF.to_csv(f"{folderPath}UNSW-NB15_1_cleaned_encoded.csv")